# 🔄 Lakeflow Job Orchestration

## Purpose
Create a Databricks Workflow (Lakeflow Job) that orchestrates the entire data pipeline from bronze ingestion to vector index sync.

## Pipeline Tasks
1. **ingest_bronze** - Run 01_ingest_bronze.ipynb
2. **transform_silver** - Run 02_transform_silver.ipynb (depends on Task 1)
3. **build_gold** - Run 03_build_gold.ipynb (depends on Task 2)
4. **sync_vector_index** - Run 04_build_vector_index.ipynb (depends on Task 3)

## Schedule
* **Frequency:** Daily at 02:00 UTC
* **On Failure:** Email notification
* **Retry Policy:** 2 retries with 5-minute delay

## Outputs
* Databricks Job ID
* Job UI link for monitoring

In [0]:
# ============================================================================
# LAKEFLOW JOB ORCHESTRATION SETUP
# ============================================================================

import json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

print("="*70)
print("🔄 DATABRICKS LAKEFLOW JOB CREATION")
print("="*70)

# Initialize Databricks client
w = WorkspaceClient()

# Get current user for email notifications
current_user = w.current_user.me()
user_email = current_user.user_name

print(f"\nCurrent user: {user_email}")

# Job configuration
JOB_NAME = "VF_Health_Ghana_IDP_Pipeline"
USER_HOME = f"/Users/{user_email}"
NOTEBOOK_BASE_PATH = f"{USER_HOME}/databricks_accenture_hackathon_virtue_foundationtrack/databricks/notebooks"

print(f"Notebook base path: {NOTEBOOK_BASE_PATH}")

# ============================================================================
# DEFINE JOB WITH 4 TASKS
# ============================================================================

# Task definitions
task_list = [
    # Task 1: Ingest Bronze
    jobs.Task(
        task_key="ingest_bronze",
        description="Ingest raw CSV data into bronze layer",
        notebook_task=jobs.NotebookTask(
            notebook_path=f"{NOTEBOOK_BASE_PATH}/01_ingest_bronze",
            source=jobs.Source.WORKSPACE
        ),
        timeout_seconds=1800,  # 30 minutes
    ),
    
    # Task 2: Transform Silver (depends on Task 1)
    jobs.Task(
        task_key="transform_silver",
        description="Clean and transform data into silver layer with intelligence fields",
        depends_on=[jobs.TaskDependency(task_key="ingest_bronze")],
        notebook_task=jobs.NotebookTask(
            notebook_path=f"{NOTEBOOK_BASE_PATH}/02_transform_silver",
            source=jobs.Source.WORKSPACE
        ),
        timeout_seconds=1800,
    ),
    
    # Task 3: Build Gold (depends on Task 2)
    jobs.Task(
        task_key="build_gold",
        description="Create gold tables for analytics and RAG",
        depends_on=[jobs.TaskDependency(task_key="transform_silver")],
        notebook_task=jobs.NotebookTask(
            notebook_path=f"{NOTEBOOK_BASE_PATH}/03_build_gold",
            source=jobs.Source.WORKSPACE
        ),
        timeout_seconds=1800,
    ),
    
    # Task 4: Build Agent (depends on Task 3)
    jobs.Task(
        task_key="build_agent",
        description="Create LangChain agent with Unity Catalog tools",
        depends_on=[jobs.TaskDependency(task_key="build_gold")],
        notebook_task=jobs.NotebookTask(
            notebook_path=f"{NOTEBOOK_BASE_PATH}/05_build_agent",
            source=jobs.Source.WORKSPACE
        ),
        timeout_seconds=2400,  # 40 minutes
    )
]

# Schedule: Daily at 02:00 UTC
schedule_config = jobs.CronSchedule(
    quartz_cron_expression="0 0 2 * * ?",  # At 02:00 UTC daily
    timezone_id="UTC",
    pause_status=jobs.PauseStatus.UNPAUSED
)

# Email notifications on failure
email_config = jobs.JobEmailNotifications(
    on_failure=[user_email],
    on_success=[],  # Only notify on failure to reduce noise
)

# Default tags
tag_dict = {
    "project": "virtue_foundation",
    "environment": "production",
    "pipeline": "ghana_healthcare_idp",
    "created_by": user_email
}

print("\n🛠️ Job Configuration:")
print(f"   Name: {JOB_NAME}")
print(f"   Tasks: 4 (bronze → silver → gold → agent)")
print(f"   Schedule: Daily at 02:00 UTC")
print(f"   Email notifications: {user_email}")
print(f"   Max concurrent runs: 1")

# ============================================================================
# CREATE OR UPDATE JOB
# ============================================================================

print("\n📦 Checking for existing job...")

# Check if job already exists
existing_jobs = w.jobs.list(name=JOB_NAME)
job_exists = False
job_id = None

for job in existing_jobs:
    if job.settings.name == JOB_NAME:
        job_exists = True
        job_id = job.job_id
        print(f"   ✅ Found existing job: {job_id}")
        break

if job_exists:
    print(f"\n🔄 Updating existing job {job_id}...")
    try:
        w.jobs.reset(
            job_id=job_id,
            new_settings=jobs.JobSettings(
                name=JOB_NAME,
                description="Virtue Foundation Ghana IDP Pipeline: Bronze → Silver → Gold → Agent",
                tasks=task_list,
                schedule=schedule_config,
                email_notifications=email_config,
                max_concurrent_runs=1,
                timeout_seconds=7200,
                tags=tag_dict
            )
        )
        print(f"   ✅ Job updated successfully")
    except Exception as e:
        print(f"   ❌ Update failed: {e}")
        raise
else:
    print("\n🆕 Creating new job...")
    try:
        created_job = w.jobs.create(
            name=JOB_NAME,
            description="Virtue Foundation Ghana IDP Pipeline: Bronze → Silver → Gold → Agent",
            tasks=task_list,
            schedule=schedule_config,
            email_notifications=email_config,
            max_concurrent_runs=1,
            timeout_seconds=7200,
            tags=tag_dict
        )
        job_id = created_job.job_id
        print(f"   ✅ Job created successfully: {job_id}")
    except Exception as e:
        print(f"   ❌ Creation failed: {e}")
        raise

# ============================================================================
# PRINT JOB DETAILS
# ============================================================================

print("\n" + "="*70)
print("✓ LAKEFLOW JOB CONFIGURED SUCCESSFULLY")
print("="*70)
print(f"\nJob ID: {job_id}")
print(f"Job Name: {JOB_NAME}")

# Get workspace URL
workspace_url = w.config.host
job_url = f"{workspace_url}/#job/{job_id}"

print(f"\n🔗 Job URL: {job_url}")
print("\n📅 Schedule: Daily at 02:00 UTC")
print(f"📧 Notifications: {user_email}")
print("\n📊 Pipeline Tasks:")
print("   1. ingest_bronze (01_ingest_bronze)")
print("   2. transform_silver (02_transform_silver) [depends on 1]")
print("   3. build_gold (03_build_gold) [depends on 2]")
print("   4. build_agent (05_build_agent) [depends on 3]")
print("\n🚀 To run immediately, click 'Run Now' in the job UI")
print("="*70)